## __Tópicos avanzados en Inteligencia Artificial 1 - MIA__

__Profesor__: Anthony D. Cho

__Ayudante__: Luis Oliveros

**Asunto**: Pytorch. Uso de Sequential
*****

## Librerias

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from time import time
from numpy import argmin, argmax
from pandas import read_csv, DataFrame
from tqdm import tqdm
import matplotlib.pyplot as plt
%matplotlib inline

## Pre-processing
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

## Metrics
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

## Pytorch
import torch
from torch.utils.data import DataLoader, TensorDataset
from torch import nn, optim

print(torch.__version__)
print("CUDA is available? ->", torch.cuda.is_available())

# Make device agnostic code
device = "cuda" if torch.cuda.is_available() else "cpu"
device

## Dataset

<center>
    <img src=https://www.ganeshdiagnostic.com/admin/public/assets/images/blog/banner/mobile/1701087153-All%20About%20Breast%20Cancer%20in%20women.webp width=800>
</center>

El conjunto de datos contiene información de 10 características de controles clínicos, específicamente de analisis de sangre, de 64 pacientes con cancer de mama y 52 pacientes sanos. 

Los 10 predictores son cuantitativos y la variable de salida (**Classification**) está etiquetada como: 1. Pacienciente sano y 2. Paciente con presencia de cancer

* La data y detalles está completamente disponible en [UCI Repository: Breast Cancer Coimbra](https://archive.ics.uci.edu/ml/datasets/Breast+Cancer+Coimbra)

#### Carga de datos

In [ ]:
## Load data
data = read_csv('https://raw.githubusercontent.com/adoc-box/Datasets/main/breast_cancer_coimbra.csv')

## Feature names list
feature_names = data.columns[:-1]
feature_names

In [ ]:
data.head(4)

In [ ]:
data['Classification'].value_counts()

## Preprocesamiento de datos

In [ ]:
## Predictors and target assignment
X = data.drop(columns=['Classification'])
y = data[['Classification']]

## Partition sets
X_trainVal, X_test, y_trainVal, y_test = train_test_split(X, y, stratify=y, random_state=84)
X_train, X_val, y_train, y_val = train_test_split(X_trainVal, y_trainVal, stratify=y_trainVal, test_size=0.15, random_state=84)

## scaling
scale = MinMaxScaler().fit(X_train)
X_train = scale.transform(X_train)
X_val = scale.transform(X_val)
X_test = scale.transform(X_test)
X_trainVal = scale.transform(X_trainVal)

## Display data shape
print('(train shape) X: {}, y: {}'.format(X_train.shape, y_train.shape))
print('(Validate shape) X: {}, y: {}'.format(X_val.shape, y_val.shape))
print('(test shape) X: {}, y: {}'.format(X_test.shape, y_test.shape))
print('(train-validate shape) X: {}, y: {}'.format(X_trainVal.shape, y_trainVal.shape))

In [ ]:
## Objetivo target: detectar pacientes con presencia de cancer. 
y_train = y_train-1
y_val = y_val-1
y_trainVal = y_trainVal-1
y_test = y_test-1

In [ ]:
BATCH_SIZE=15

## Convertir los datos en tensores
X_train_tensor = torch.FloatTensor(X_train) 
X_val_tensor = torch.FloatTensor(X_val) 
X_test_tensor = torch.FloatTensor(X_test) 
X_trainVal_tensor = torch.FloatTensor(X_trainVal) 

y_train_tensor = torch.FloatTensor(y_train.values)
y_val_tensor = torch.FloatTensor(y_val.values)
y_test_tensor = torch.FloatTensor(y_test.values)
y_trainVal_tensor = torch.FloatTensor(y_trainVal.values)


## Crear un gestor de datos de pytorch
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
trainVal_dataset = TensorDataset(X_trainVal_tensor, y_trainVal_tensor)

train_dataloader = DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_dataloader = DataLoader(dataset=val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_dataloader = DataLoader(dataset=test_dataset, batch_size=BATCH_SIZE, shuffle=False)
trainVal_dataloader = DataLoader(dataset=trainVal_dataset, batch_size=BATCH_SIZE, shuffle=True)

### Diseño del modelo

In [ ]:
## Model instance
model = nn.Sequential()
model.add_module(name='Dense_01', module=nn.Linear(in_features=X_train.shape[1], 
                                                   out_features=64))
model.add_module(name='Act_01', module=nn.ReLU())
model.add_module(name='Dense_02', module=nn.Linear(in_features=64, 
                                                   out_features=16))
model.add_module(name='Act_02', module=nn.ReLU())
model.add_module(name='output', module=nn.Linear(in_features=16, 
                                                   out_features=1))
model.add_module(name='Act_output', module=nn.Sigmoid())

## Compiler setting
loss_fn = nn.BCELoss() ## Binary Cross Entropy
optimizer = optim.Adam(params=model.parameters(), lr=0.001)

## Model summary
print(model)

In [ ]:
NUM_EPOCHS = 200

## Performance allocation
performance = {'loss': [], 'val_loss': [], 
               'accuracy': [], 'val_accuracy': []}

start = time()
for epoch in range(NUM_EPOCHS):
    
    ## Set model to training mode
    model.train()

    ## cumulated loss
    loss_train_cum, loss_val_cum = 0, 0
    y_pred, y_true = [], []

    for (x_batch, y_batch) in train_dataloader:

        ## Compute prediction from model
        y_batch_pred = model(x_batch)

        ## Compute loss 
        loss = loss_fn(y_batch_pred, y_batch)

        ## Clear gradient, apply propagation, and model weight updating
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        ## Store cumulated loss
        loss_train_cum += loss.item() * x_batch.size(0)

        ## Store predicted and ground truth labels
        y_pred.extend(y_batch_pred.flatten().round().detach().numpy())
        y_true.extend(y_batch.flatten().detach().numpy())
    
    ## Compute epoch loss and metric
    epoch_train_loss = loss_train_cum / len(train_dataloader.dataset)
    epoch_train_accuracy = accuracy_score(y_pred=y_pred, y_true=y_true)
    performance['loss'].append(epoch_train_loss)
    performance['accuracy'].append( epoch_train_accuracy )

    ## Set model to evaluate mode
    model.eval()

    with torch.no_grad():

        y_pred, y_true = [], []
        for (x_batch, y_batch) in val_dataloader:

            ## Compute prediction from model
            y_batch_pred = model(x_batch)

            ## Compute loss 
            loss_val = loss_fn(y_batch_pred, y_batch)

            ## Store cumulated loss
            loss_val_cum += loss_val.item() * x_batch.size(0)

            ## Store predicted and ground truth labels
            y_pred.extend(y_batch_pred.flatten().round().numpy())
            y_true.extend(y_batch.flatten().numpy())

    ## Compute epoch loss and metric
    epoch_val_loss = loss_val_cum / len(val_dataloader.dataset)
    epoch_val_accuracy = accuracy_score(y_pred=y_pred, y_true=y_true)
    performance['val_loss'].append(epoch_val_loss)
    performance['val_accuracy'].append( epoch_val_accuracy )

    print('Epoch {:4}/{}, loss: {:.4f}, accuracy: {:.4f}, val_loss: {:.4f}, val_accuracy: {:.4f}'.format(
                                                                epoch+1, NUM_EPOCHS,
                                                                epoch_train_loss, epoch_train_accuracy,
                                                                epoch_val_loss, epoch_val_accuracy))

    ## Clear gradient
    optimizer.zero_grad()

stop = time()
print('Time spent[s]: {:2f}'.format(stop -start))

In [ ]:
DataFrame(performance).plot(figsize=(15, 4))

In [ ]:
## Searching for the best epoch
id_min = argmin(performance['val_loss'])
print('Loss - Validation: {} - Error: {}'.format(id_min+1, performance['val_loss'][id_min]))

id_max = argmax(performance['val_accuracy'])
print('Accuracy - Validation: {} - Error: {}'.format(id_max+1, performance['val_accuracy'][id_max]))

## Mejor modelo

In [ ]:
## Model instance
model = nn.Sequential()
model.add_module(name='Dense_01', module=nn.Linear(in_features=X_train.shape[1], 
                                                   out_features=64))
model.add_module(name='Act_01', module=nn.ReLU())
model.add_module(name='Dense_02', module=nn.Linear(in_features=64, 
                                                   out_features=16))
model.add_module(name='Act_02', module=nn.ReLU())
model.add_module(name='output', module=nn.Linear(in_features=16, 
                                                   out_features=1))
model.add_module(name='Act_output', module=nn.Sigmoid())

## Compiler setting
loss_fn = nn.BCELoss() ## Binary Cross Entropy
optimizer = optim.Adam(params=model.parameters(), lr=0.001)

In [ ]:
NUM_EPOCHS = 150

start = time()
for epoch in range(NUM_EPOCHS):
    
    ## Set model to training mode
    model.train()

    ## cumulated loss
    loss_train_cum, loss_val_cum = 0, 0
    y_pred, y_true = [], []

    for (x_batch, y_batch) in trainVal_dataloader:

        ## Compute prediction from model
        y_batch_pred = model(x_batch)

        ## Compute loss 
        loss = loss_fn(y_batch_pred, y_batch)

        ## Clear gradient, apply propagation, and model weight updating
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        ## Store cumulated loss
        loss_train_cum += loss.item() * x_batch.size(0)

        ## Store predicted and ground truth labels
        y_pred.extend(y_batch_pred.flatten().round().detach().numpy())
        y_true.extend(y_batch.flatten().detach().numpy())
    
    ## Compute epoch loss and metric
    epoch_train_loss = loss_train_cum / len(train_dataloader.dataset)
    epoch_train_accuracy = accuracy_score(y_pred=y_pred, y_true=y_true)

    print('Epoch {:4}/{}, loss: {:.4f}, accuracy: {:.4f}'.format(
                                                                epoch+1, NUM_EPOCHS,
                                                                epoch_train_loss, epoch_train_accuracy,
                                                                ))

stop = time()
print('Time spent[s]: {:2f}'.format(stop -start))

In [ ]:
## Set model to evaluate mode
model.eval()

## Compute prediction
with torch.no_grad():
    prediction = model(X_test_tensor)
prediction

In [ ]:
## Convert tensor to array and decodifying prediction to class
predictionClass = prediction.flatten().detach().numpy().round()
predictionClass

In [ ]:
## Display confusion matrix
cm = confusion_matrix(y_true=y_test, y_pred=predictionClass)
CM = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['1','2'])
CM.plot();

In [ ]:
## Display classification report
print(classification_report(y_true=y_test, 
                            y_pred=predictionClass, 
                            target_names=['1', '2']
                            ))